# Protein–Protein Contact Network Analysis

Builds residue–residue contact networks for a homo-oligomeric ion channel and
identifies the pairs whose contacts respond to a change in conditions.

| Output | What it measures |
|---|---|
| Distance matrix | Mean residue–residue distance, averaged over frames |
| Contact probability matrix | Fraction of frames each residue pair is within the cutoff |
| Block-averaged error | Standard error of each contact probability, from block averaging |
| Intra / adjacent / diagonal decomposition | Contacts within a subunit vs between neighbouring vs opposite subunits |
| C4-symmetry-filtered contacts | Contacts that are reproduced in every subunit; asymmetric ones are removed before any figure is built |
| Contact variance maps | Which pairs change most across a set of conditions |
| Monotonic-trend pairs | Pairs that consistently gain or lose contact as temperature rises |
| Arc diagrams | The most variable pairs drawn along the sequence, shaded and weighted by their variance |
| Per-pair comparisons | Contact probability of chosen pairs across conditions, with error bars |

The pairs that gain or lose contact monotonically with temperature are the
interesting output: in a thermosensitive channel those are candidates for the
contacts that break as the gate opens.

---

## The subunit decomposition

For a homo-oligomer, a contact matrix over the whole assembly contains the same
information many times over. With `N_PROTOMERS = 4` the matrix is reshaped into
subunit blocks and averaged into three physically distinct classes:

- **intra** — residue *i* of a subunit with residue *j* of the *same* subunit.
  Internal architecture of one protomer.
- **adjacent** — subunit *k* with subunit *k±1*. The main subunit–subunit
  interface.
- **diagonal** — subunit *k* with subunit *k+2*: diagonally opposite. For a
  tetramer these are the contacts across the pore.

For a tetramer each subunit has two adjacent neighbours and one diagonal one, so
an "any other subunit" average is `(2 × adjacent + diagonal) / 3`. That weighting
is computed by `inter_subunit_average()` from `N_PROTOMERS` rather than hard
coded, so it stays correct for a trimer or a pentamer.

## The symmetry filter

A homo-oligomer is a symmetric object, and a residue–residue contact that only
exists in one of its four subunits is far more likely to be a sampling artefact
than a property of the channel. Left in, such contacts dominate the
temperature-variance analysis: in the unfiltered analysis they carried 57%
(atomistic) and 59% (coarse-grained) of the variance in the temperature-variance
figure, and every one of its twelve strongest hot spots, at both resolutions,
was present in only one of the four subunits.

The notebook therefore applies a **C4 (Cn) symmetry filter to the assembly-wide
contact matrix before anything else is computed from it**. For every residue
pair *(i, j)* and every subunit relation (same subunit, neighbouring subunit,
opposite subunit) the four symmetric copies of that contact are compared:

- present in **all** copies → kept, and averaged as before;
- absent in **all** copies → kept (symmetric absence is fine);
- present in **some but not all** copies → *asymmetric*, and removed from every
  copy.

"Present" means a contact probability of at least `SYMMETRY_MIN_PROBABILITY`;
`SYMMETRY_MIN_COPIES` sets how many of the `N_PROTOMERS` copies must show the
contact (default: all of them). Every downstream figure — contact maps,
variance maps, arc diagrams, trend detection and the per-pair comparisons — is
built from the same filtered set, so they are mutually consistent. The filter
can be switched off with `APPLY_SYMMETRY_FILTER = False` for comparison; the
unfiltered assembly matrices are always kept in `network[level]["analysis"]`.

## Expected data layout

```
<DATA_ROOT>/
├── coarse/
│   ├── 305K05/
│   │   ├── NPTProteinCenter.gro
│   │   └── NPTProteinCenter.xtc
│   └── ...
└── atomistic/
    └── ...
```

## Generating the input files

```bash
gmx trjconv -s npt.tpr -f npt.xtc -o NPTProteinCenter.xtc \
            -pbc mol -center -n index.ndx     # select the protein group
gmx trjconv -s npt.tpr -f NPTProteinCenter.xtc -o NPTProteinCenter.gro -dump 0
```

A protein-only trajectory is fine here, and keeps these calculations affordable.

## Error bars come from block averaging

Consecutive MD frames are correlated, so the naive standard error over frames is
much too small. The trajectory is instead split into `N_BLOCKS` contiguous
blocks, the contact probability computed per block, and the standard error taken
across block means. This is only meaningful with enough blocks: **use at least
5, preferably 10**. The original analysis used 2, which cannot estimate a
spread — that default has been raised here.

## Units

Distances are in **ångström** (MDAnalysis' convention), not GROMACS' nm.

## Adapting to a different channel — checklist

1. `N_PROTOMERS` — drives the whole subunit decomposition.
2. `CONTACT_CUTOFF` — per model level, as in the lipid notebook.
3. `PROTEIN_SELECTION` / `DISTANCE_SELECTION` — the latter picks one
   representative atom per residue (`name CA` all-atom, `name BB` for Martini).
4. `N_BLOCKS` — how many blocks the error estimate uses.
5. `SYMMETRY_MIN_PROBABILITY`, `SYMMETRY_MIN_COPIES` — what counts as a
   contact being "present" in a subunit, and in how many subunits it must
   be present to survive the symmetry filter.
6. `RESIDUE_PAIRS_OF_INTEREST` — specific pairs to compare in detail.
7. `DOMAIN_SEGMENTS`, `RESIDUE_OFFSET` — as in the other notebooks.


---

## 1. Imports

In [ ]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import os
import re
import warnings
import itertools
from itertools import combinations
from dataclasses import dataclass
from pathlib import Path

# ---------------------------------------------------------------------------
# Third party
# ---------------------------------------------------------------------------
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize     # colour scaling
from matplotlib.colors import LinearSegmentedColormap  # grey ramp for arc diagrams
from matplotlib.patches import Arc, Patch            # arc diagrams, legend handles
from matplotlib.ticker import ScalarFormatter        # compact axis labels

import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array, capped_distance
from scipy.signal import savgol_filter               # smoothing of contact time series

# MDAnalysis warns about topology attributes a .gro file legitimately lacks
# (masses, elements). Silence only that, rather than filtering all warnings -
# a blanket filter also hides real numerical problems.
warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis")

# ---------------------------------------------------------------------------
# Shared repository configuration
# ---------------------------------------------------------------------------
# channel_config.py sits beside this notebook and holds everything common to
# every notebook here: the data root, the simulation grid, the channel topology,
# the domain boundaries and the plotting primitives. The star import brings in
# only the shared, read-only names it lists in __all__.
import channel_config as cc
from channel_config import *

%matplotlib inline

print(f"numpy {np.__version__}")
print(f"MDAnalysis {mda.__version__}")

---

## 2. Configuration

Everything shared with the other notebooks lives in `channel_config.py`; only this notebook's own settings are here.

In [ ]:
# ===========================================================================
#                            C O N F I G U R A T I O N
#
#  Only the settings specific to THIS notebook live here. Everything shared
#  across the repository - the data root, the simulation grid, the channel
#  topology, the domain boundaries and the plot style - lives in
#  channel_config.py, so it can only ever be defined once.
#
#  To change a shared value (protomer count, temperatures, domains, ...), edit
#  channel_config.py and every notebook picks it up.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1. This notebook
# ---------------------------------------------------------------------------
# Set USE_DEMO_DATA = True to generate small synthetic inputs and run the whole
# notebook without real trajectories. Set it to False to use your own data;
# point channel_config.REAL_DATA_ROOT at it, or set the MD_DATA_ROOT
# environment variable.
USE_DEMO_DATA = True

# Names this notebook's output directory, results/<NOTEBOOK_SLUG>/. Change it if
# you copy this notebook as the basis for a new analysis, so the two cannot
# overwrite each other's figures.
NOTEBOOK_SLUG = "protein_protein_network"

# Write every figure to OUTPUT_DIR as it is drawn.
SAVE_FIGURES = False

# Logical name -> file name inside each condition directory.
FILENAMES = {
    "topology": "NPTProteinCenter.gro",
    "trajectory": "NPTProteinCenter.xtc",
}

# Resolved paths.
DATA_ROOT = cc.resolve_data_root(USE_DEMO_DATA)
DEMO_DATA_ROOT = cc.DEMO_DATA_ROOT
OUTPUT_DIR = cc.make_output_dir(NOTEBOOK_SLUG)


def build_path(model_level, temperature_k, salinity_mm, key):
    """Return the full path to one input file.

    Override this if your directory tree is shaped differently, e.g. if the
    model level is a filename suffix rather than a parent directory. The
    condition directory name itself comes from cc.condition_dirname().
    """
    return (
        DATA_ROOT
        / model_level
        / cc.condition_dirname(temperature_k, salinity_mm)
        / FILENAMES[key]
    )


def save_figure(fig, name):
    """Save a figure into this notebook's output directory."""
    return cc.save_figure(fig, name, OUTPUT_DIR, SAVE_FIGURES)


# ---------------------------------------------------------------------------
# 2. Selections and contact definition
# ---------------------------------------------------------------------------
# How the protein is selected. Coarse-grained models often fall outside
# MDAnalysis' notion of "protein"; use e.g. "name BB SC1 SC2 SC3 SC4" if so.
PROTEIN_SELECTION = {
    "coarse": "protein",
    "atomistic": "protein",
}

# One representative atom per residue, for the residue-residue DISTANCE matrix.
# The contact matrix uses all atoms of PROTEIN_SELECTION instead.
DISTANCE_SELECTION = {
    "coarse": "name BB",
    "atomistic": "name CA",
}

# Contact cutoff (A). A coarse bead stands for several heavy atoms, so it needs a
# larger cutoff than an all-atom heavy atom.
CONTACT_CUTOFF = {
    "coarse": 6.0,
    "atomistic": 4.5,
}

HEAVY_ATOMS_ONLY = {
    "coarse": False,
    "atomistic": True,
}

# ---------------------------------------------------------------------------
# 3. Frame selection and error estimation
# ---------------------------------------------------------------------------
# The demo trajectories are only a few dozen frames long, so a stride meant for
# a real run would leave nothing to analyse.
FRAME_STEP = 1 if USE_DEMO_DATA else 50
START_FRAME = 0
END_FRAME = None          # None -> to the end of each trajectory

# Number of contiguous blocks used for the block-averaged standard error.
# Consecutive frames are correlated, so a naive standard error over frames is far
# too small. Use at least 5 blocks; 10 or more is better. Two blocks cannot
# estimate a spread at all.
N_BLOCKS = 5 if USE_DEMO_DATA else 10

# ---------------------------------------------------------------------------
# 4. Symmetry filter
# ---------------------------------------------------------------------------
# The channel is a homo-oligomer, so every residue-residue contact has
# N_PROTOMERS symmetry-equivalent copies (one per subunit for an intra-subunit
# contact, one per subunit pair for an interface contact). A contact that shows
# up in only some of those copies is asymmetric and is treated as a sampling
# artefact rather than a feature of the channel: it is removed from ALL copies
# before any averaging, variance or trend calculation. Every figure below is
# then built from the same filtered contact set.
#
# Set False to reproduce the unfiltered analysis for comparison.
APPLY_SYMMETRY_FILTER = True

# A contact counts as "present" in a subunit copy when its probability is at
# least this. Zero would make a single frame's contact count as present; a
# small positive value asks for the contact to be more than a fluke.
SYMMETRY_MIN_PROBABILITY = 0.05

# In how many of the N_PROTOMERS copies the contact must be present to be kept.
# None -> all of them (the strict C4 criterion). A contact present in none of
# the copies is symmetric too (symmetric absence) and is never touched.
SYMMETRY_MIN_COPIES = None

# ---------------------------------------------------------------------------
# 5. Contact-variance and trend detection
# ---------------------------------------------------------------------------
# Minimum variance for a pair to be drawn in the arc diagrams. Raise it to
# declutter: the number of pairs above the threshold grows very quickly.
VARIANCE_THRESHOLD = 1.0e-3

# Minimum sequence separation |i - j|. Neighbours along the chain are in contact
# by construction and only add noise.
MIN_SEQUENCE_SEPARATION = 4

# Monotonic-trend detection (see the trend cell for what these mean).
TREND_TAU_MIN = 0.75          # |Kendall tau| against temperature
TREND_MIN_CHANGE = 0.05       # minimum change in probability across the span
TREND_P_FLOOR = 0.02          # drop pairs that essentially never form
TREND_P_CEILING = 0.98        # drop pairs that essentially never break

# ---------------------------------------------------------------------------
# 6. Specific residue pairs to compare in detail
# ---------------------------------------------------------------------------
# Given as REAL RESIDUE NUMBERS on the common numbering scheme, as pairs.
# Set to None to have the notebook pick the strongest-trending pairs instead,
# which avoids the usual failure mode of a hand-maintained list going stale
# against the data.
RESIDUE_PAIRS_OF_INTEREST = None

# How many pairs to show when RESIDUE_PAIRS_OF_INTEREST is None.
N_AUTO_PAIRS = 12


# ---------------------------------------------------------------------------
# Apply the shared plot style and report the resolved configuration
# ---------------------------------------------------------------------------
cc.apply_style()
cc.describe_config(DATA_ROOT, USE_DEMO_DATA, OUTPUT_DIR)

---

## 3. The condition grid

In [ ]:
# The set of simulations, built once from the shared grid definition. Every
# later selection is a query against it - grid.where(salinity_mm=50) returns the
# positions of those runs - so slicing is never hard coded as [0:6] or [12:18].
grid = cc.ConditionGrid(TEMPERATURES_K, SALINITIES_MM, build_path)
grid.describe()

---

## 4. Demo data (optional)

In [ ]:
# ===========================================================================
#  Optional: synthetic demo data
#
#  Builds a small four-subunit protein and writes .gro/.xtc per condition.
#
#  A few residue pairs are given a contact that weakens with temperature, so the
#  trend detection has a real signal to recover rather than pure noise. All of
#  it is fabricated.
# ===========================================================================

DEMO_RESIDUES_PER_PROTOMER = 40
DEMO_N_FRAMES = 30
DEMO_BOX = 120.0
DEMO_RING_RADIUS = 16.0

# (residue_a, residue_b) 0-based within a protomer. These are pulled together at
# low temperature and released at high temperature.
DEMO_BREAKING_PAIRS = ((6, 30), (11, 27), (18, 35))
# These do the opposite: they form as temperature rises.
DEMO_FORMING_PAIRS = ((3, 21), (14, 33))
# Interface contacts: residue a of one subunit against residue b of the NEXT
# subunit, so the inter-subunit ("adjacent") class has signal in it too.
DEMO_INTERFACE_PAIRS = ((5, 15),)
# An ASYMMETRIC contact, planted in the first protomer only, that also changes
# with temperature. Unfiltered it would rank among the most variable pairs;
# the symmetry filter should remove it, which is how the filter is tested here.
DEMO_ASYMMETRIC_PAIRS = ((8, 24),)

DEMO_ATOM_NAMES = {
    "coarse": ("BB", "SC1"),
    "atomistic": ("N", "CA", "C", "O"),
}


def _build_demo_universe(model_level):
    """Assemble a four-subunit protein Universe for one model level."""
    atom_names = DEMO_ATOM_NAMES[model_level]
    atoms_per_residue = len(atom_names)
    n_res = DEMO_RESIDUES_PER_PROTOMER * N_PROTOMERS
    n_atoms = n_res * atoms_per_residue

    universe = mda.Universe.empty(
        n_atoms, n_residues=n_res,
        atom_resindex=np.repeat(np.arange(n_res), atoms_per_residue),
        trajectory=True)
    universe.add_TopologyAttr("name", list(atom_names) * n_res)
    universe.add_TopologyAttr("type", [n[0] for n in atom_names] * n_res)
    universe.add_TopologyAttr("resname", ["ALA"] * n_res)
    # Numbering chosen so that adding RESIDUE_OFFSET puts each model level onto
    # the common scheme, as real data from either resolution would.
    first_residue = RESIDUE_NUMBER_ORIGIN - RESIDUE_OFFSET.get(model_level, 0)
    universe.add_TopologyAttr(
        "resid", list(range(first_residue, first_residue + n_res)))
    universe.add_TopologyAttr("segid", ["A"])
    universe.dimensions = [DEMO_BOX] * 3 + [90.0] * 3
    return universe, atoms_per_residue


def _demo_positions(seed, temperature_k, atoms_per_residue):
    """Positions for one frame: four helical subunits around a central axis."""
    rng = np.random.default_rng(seed)
    heat = (temperature_k - min(TEMPERATURES_K)) / 25.0
    centre = DEMO_BOX / 2.0
    n_res_per_prot = DEMO_RESIDUES_PER_PROTOMER

    residue_centres = np.empty((n_res_per_prot * N_PROTOMERS, 3))
    index = 0
    for protomer in range(N_PROTOMERS):
        base = protomer * 2.0 * np.pi / N_PROTOMERS
        for residue in range(n_res_per_prot):
            angle = base + 0.09 * residue
            radius = DEMO_RING_RADIUS + 2.0 * np.sin(0.3 * residue)
            residue_centres[index] = [
                centre + radius * np.cos(angle),
                centre + radius * np.sin(angle),
                centre + (residue - n_res_per_prot / 2.0) * 1.1,
            ]
            index += 1

    # Impose the planted contacts by pulling the partner residue close.
    for protomer in range(N_PROTOMERS):
        offset = protomer * n_res_per_prot
        for a, b in DEMO_BREAKING_PAIRS:
            # Fully formed at the lowest temperature, gone at the highest.
            gap = 3.0 + 9.0 * heat
            direction = rng.normal(0, 1, 3)
            direction /= np.linalg.norm(direction)
            residue_centres[offset + b] = residue_centres[offset + a] + gap * direction
        for a, b in DEMO_FORMING_PAIRS:
            gap = 12.0 - 9.0 * heat
            direction = rng.normal(0, 1, 3)
            direction /= np.linalg.norm(direction)
            residue_centres[offset + b] = residue_centres[offset + a] + gap * direction

    # Interface contacts, placed after the intra-subunit ones so they are not
    # overwritten. Each target residue is written exactly once.
    for protomer in range(N_PROTOMERS):
        neighbour = (protomer + 1) % N_PROTOMERS
        for a, b in DEMO_INTERFACE_PAIRS:
            gap = 3.0 + 9.0 * heat
            direction = rng.normal(0, 1, 3)
            direction /= np.linalg.norm(direction)
            residue_centres[neighbour * n_res_per_prot + b] = (
                residue_centres[protomer * n_res_per_prot + a] + gap * direction)

    # The asymmetric contact: written for protomer 0 only, so its three
    # symmetry mates never form it. Placed last so nothing overwrites it.
    for a, b in DEMO_ASYMMETRIC_PAIRS:
        gap = 3.0 + 9.0 * heat
        direction = rng.normal(0, 1, 3)
        direction /= np.linalg.norm(direction)
        residue_centres[b] = residue_centres[a] + gap * direction

    residue_centres += rng.normal(0.0, 0.4, residue_centres.shape)
    positions = np.repeat(residue_centres, atoms_per_residue, axis=0)
    positions += rng.normal(0.0, 0.6, positions.shape)
    return positions % DEMO_BOX


def generate_demo_dataset():
    """Write a .gro/.xtc pair for every condition at every model level."""
    written = 0
    for model_level in MODEL_LEVELS:
        universe, atoms_per_residue = _build_demo_universe(model_level)
        for cond in grid:
            topology = build_path(model_level, cond.temperature_k,
                                  cond.salinity_mm, "topology")
            trajectory = build_path(model_level, cond.temperature_k,
                                    cond.salinity_mm, "trajectory")
            topology.parent.mkdir(parents=True, exist_ok=True)

            universe.atoms.positions = _demo_positions(
                0, cond.temperature_k, atoms_per_residue)
            universe.atoms.write(str(topology))
            with mda.Writer(str(trajectory), n_atoms=len(universe.atoms)) as writer:
                for frame in range(DEMO_N_FRAMES):
                    universe.atoms.positions = _demo_positions(
                        frame + 1, cond.temperature_k, atoms_per_residue)
                    writer.write(universe.atoms)
            written += 2
    return written


if USE_DEMO_DATA:
    n_written = generate_demo_dataset()
    origin = RESIDUE_NUMBER_ORIGIN
    print(f"Wrote {n_written} synthetic files under {DEMO_DATA_ROOT.resolve()}")
    print(f"System: {DEMO_RESIDUES_PER_PROTOMER} residues x {N_PROTOMERS} protomers, "
          f"{DEMO_N_FRAMES} frames per run")
    print("\nPlanted signals (real residue numbers), to be recovered below:")
    print("  breaking with temperature: "
          + ", ".join(f"{a + origin}-{b + origin}" for a, b in DEMO_BREAKING_PAIRS))
    print("  forming with temperature:  "
          + ", ".join(f"{a + origin}-{b + origin}" for a, b in DEMO_FORMING_PAIRS))
    print("  breaking at the subunit interface: "
          + ", ".join(f"{a + origin}-{b + origin}"
                      for a, b in DEMO_INTERFACE_PAIRS))
    print("  ASYMMETRIC (protomer 1 only; the symmetry filter should remove it): "
          + ", ".join(f"{a + origin}-{b + origin}"
                      for a, b in DEMO_ASYMMETRIC_PAIRS))
    print("\nThese files are for testing the pipeline only - not simulation data.")
else:
    for model_level in MODEL_LEVELS:
        for key in ("topology", "trajectory"):
            missing = grid.report_missing(model_level, key)
            if missing:
                print(f"  WARNING {model_level}/{key}: {len(missing)} file(s) "
                      f"missing, first is {missing[0]}")

---

## 5. Network analysis

In [ ]:
# ===========================================================================
#  Contact and distance networks
# ===========================================================================

class NetworkAnalysis:
    """Residue-residue distance and contact networks for a set of simulations.

    Parameters
    ----------
    topologies, trajectories : sequence of path
        Matched lists, one entry per simulation.
    labels : sequence of str, optional
        One per simulation, used in messages and legends.
    frame_step, start_frame, end_frame :
        Frame selection, defaulting to the configuration cell.
    """

    TOPOLOGY_SUFFIXES = {".gro", ".pdb", ".tpr"}
    TRAJECTORY_SUFFIXES = {".xtc", ".trr", ".dcd"}

    def __init__(self, topologies, trajectories, labels=None,
                 frame_step=None, start_frame=None, end_frame=None):
        if isinstance(topologies, (str, Path)):
            topologies = [topologies]
        if isinstance(trajectories, (str, Path)):
            trajectories = [trajectories]
        self.topologies = [Path(p) for p in topologies]
        self.trajectories = [Path(p) for p in trajectories]

        if not self.topologies:
            raise ValueError("No topologies given.")
        if len(self.topologies) != len(self.trajectories):
            raise ValueError(
                f"Got {len(self.topologies)} topologies but "
                f"{len(self.trajectories)} trajectories; they must correspond.")
        for paths, suffixes, what in (
            (self.topologies, self.TOPOLOGY_SUFFIXES, "topology"),
            (self.trajectories, self.TRAJECTORY_SUFFIXES, "trajectory"),
        ):
            bad = [p for p in paths if p.suffix.lower() not in suffixes]
            if bad:
                raise ValueError(
                    f"Unsupported {what} type '{bad[0].suffix}'. "
                    f"Supported: {sorted(suffixes)}")
        missing = [p for p in self.topologies + self.trajectories if not p.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} input file(s) not found. First: {missing[0]}\n"
                "Check DATA_ROOT and build_path() in the configuration cell.")

        self.labels = (list(labels) if labels is not None
                       else [p.parent.name for p in self.topologies])
        self.frame_step = FRAME_STEP if frame_step is None else frame_step
        self.start_frame = START_FRAME if start_frame is None else start_frame
        self.end_frame = END_FRAME if end_frame is None else end_frame

        self.universes = [mda.Universe(str(t), str(x))
                          for t, x in zip(self.topologies, self.trajectories)]

    def _frame_bounds(self, universe):
        """Resolve the stop frame for one universe.

        Resolved per universe rather than once: the original code assigned the
        first trajectory's length to `end_frame` and then reused that value for
        every later universe, silently truncating longer runs.
        """
        stop = len(universe.trajectory) if self.end_frame is None else self.end_frame
        return self.start_frame, stop

    # -- distance network ---------------------------------------------------
    def distance_network(self, selection):
        """Mean pairwise distance between selected atoms, averaged over frames.

        One representative atom per residue is expected (`name CA` or `name BB`),
        giving an (n_residues, n_residues) matrix in angstrom.

        Periodic boundaries are applied. The original implementation omitted the
        box, so any pair separated across a periodic boundary was reported at its
        image distance instead of the true minimum-image distance.

        Sets `self.distances`. Returns self.
        """
        self.distances = []
        for universe, label in zip(self.universes, self.labels):
            atoms = universe.select_atoms(selection)
            if len(atoms) == 0:
                raise ValueError(
                    f"{label}: the selection '{selection}' matched no atoms. "
                    "Set DISTANCE_SELECTION for this model level.")
            start, stop = self._frame_bounds(universe)
            n_frames = len(range(start, stop, self.frame_step))
            if n_frames == 0:
                raise ValueError(
                    f"{label}: frame selection [{start}:{stop}:{self.frame_step}] "
                    "is empty.")

            accumulator = np.zeros((len(atoms), len(atoms)), dtype=np.float64)
            for ts in universe.trajectory[start:stop:self.frame_step]:
                positions = atoms.positions
                accumulator += distance_array(positions, positions,
                                              box=ts.dimensions)
            self.distances.append((accumulator / n_frames).astype(np.float32))
        return self

    # -- contact network ----------------------------------------------------
    def contact_network(self, selection, cutoff, heavy_atoms_only=True,
                        n_blocks=None, set_diagonal=True):
        """Contact probability per residue pair, with a block-averaged error.

        A pair counts as in contact in a frame if *any* atom of residue i is
        within `cutoff` of any atom of residue j. `capped_distance` returns only
        the pairs inside the cutoff, so cost scales with the number of contacts
        rather than with n_atoms squared.

        The error is the standard error across `n_blocks` contiguous blocks of
        frames. Blocks, rather than individual frames, because consecutive MD
        frames are correlated and a per-frame standard error would be far too
        small. With fewer than 5 blocks the estimate is not trustworthy and a
        warning is issued.

        `set_diagonal` writes 1.0 onto the diagonal, since a residue is trivially
        in contact with itself; it makes the maps easier to read but is cosmetic.

        Sets `self.contacts` and `self.contact_errors`. Returns self.
        """
        n_blocks = N_BLOCKS if n_blocks is None else n_blocks
        if cutoff <= 0:
            raise ValueError("cutoff must be positive.")

        self.contacts = []
        self.contact_errors = []
        for universe, label in zip(self.universes, self.labels):
            protein = universe.select_atoms(selection)
            if heavy_atoms_only:
                protein = protein.select_atoms("not name H*")
            if len(protein) == 0:
                raise ValueError(
                    f"{label}: the selection '{selection}' matched no atoms. "
                    "Set PROTEIN_SELECTION for this model level.")

            residues = protein.residues
            n_res = len(residues)
            start, stop = self._frame_bounds(universe)
            n_frames = len(range(start, stop, self.frame_step))
            if n_frames < 2:
                raise ValueError(
                    f"{label}: need at least 2 frames, got {n_frames}. "
                    "Lower FRAME_STEP or widen the frame range.")

            n_blocks_used = int(min(n_blocks, n_frames))
            if n_blocks_used < 5:
                print(f"  warning: {label} uses only {n_blocks_used} block(s); "
                      "the error estimate is unreliable below about 5")
            block_size = max(1, n_frames // n_blocks_used)

            # Map atoms to positions within this residue selection.
            atom_to_residue = np.searchsorted(residues.resindices,
                                              protein.resindices)

            total = np.zeros((n_res, n_res), dtype=np.float64)
            block_totals = np.zeros((n_blocks_used, n_res, n_res), dtype=np.float64)
            block_lengths = np.zeros(n_blocks_used, dtype=np.int64)

            for iframe, ts in enumerate(
                    universe.trajectory[start:stop:self.frame_step]):
                block = min(iframe // block_size, n_blocks_used - 1)
                block_lengths[block] += 1

                pairs = capped_distance(
                    protein.positions, protein.positions, max_cutoff=cutoff,
                    box=ts.dimensions, return_distances=False)
                if len(pairs) == 0:
                    continue
                residue_pairs = atom_to_residue[pairs]
                # Drop self-contacts, then collapse to unique unordered pairs so
                # a residue pair with many atom contacts still counts once.
                residue_pairs = residue_pairs[
                    residue_pairs[:, 0] != residue_pairs[:, 1]]
                if len(residue_pairs) == 0:
                    continue
                residue_pairs = np.unique(np.sort(residue_pairs, axis=1), axis=0)
                i, j = residue_pairs[:, 0], residue_pairs[:, 1]
                total[i, j] += 1
                total[j, i] += 1
                block_totals[block, i, j] += 1
                block_totals[block, j, i] += 1

            probability = total / n_frames

            # Guard against an empty block, which would divide by zero.
            nonempty = block_lengths > 0
            if nonempty.sum() > 1:
                block_probability = (block_totals[nonempty]
                                     / block_lengths[nonempty, None, None])
                error = (np.std(block_probability, axis=0, ddof=1)
                         / np.sqrt(nonempty.sum()))
            else:
                error = np.zeros_like(probability)

            if set_diagonal:
                np.fill_diagonal(probability, 1.0)
            np.fill_diagonal(error, 0.0)

            self.contacts.append(probability.astype(np.float32))
            self.contact_errors.append(error.astype(np.float32))
        return self

    def residue_numbers(self, index=0, selection="protein",
                        heavy_atoms_only=True, residue_offset=0):
        """Real residue numbers for the matrix axes of one simulation."""
        atoms = self.universes[index].select_atoms(selection)
        if heavy_atoms_only:
            atoms = atoms.select_atoms("not name H*")
        return atoms.residues.resids + residue_offset

    # -- time series for one pair -------------------------------------------
    def residue_pair_time_series(self, residue_a, residue_b, selection,
                                 n_protomers=None, cutoff=5.0, beta=5.0,
                                 lambda_constant=1.8):
        """Track one residue pair's contact through time, averaged over subunits.

        `residue_a` and `residue_b` are 0-based indices *within a protomer*; the
        equivalent pair in every subunit is measured and averaged.

        Rather than a hard cutoff, a smooth switching function

            1 / (1 + exp(beta * (d - lambda * cutoff)))

        is used, so that a pair hovering at the cutoff produces a continuous
        signal rather than a flickering 0/1. `lambda_constant` scales where the
        switch sits relative to `cutoff`; `beta` controls its sharpness.

        Returns a dict with 'times' (ps, as recorded in the trajectory) and
        'contacts', each a list of one array per simulation.
        """
        n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
        series, times = [], []

        for universe, label in zip(self.universes, self.labels):
            protein = universe.select_atoms(selection)
            residues = protein.residues
            per_protomer = len(residues) // n_protomers
            if len(residues) % n_protomers != 0:
                raise ValueError(
                    f"{label}: {len(residues)} residues is not divisible by "
                    f"n_protomers={n_protomers}.")
            if not (0 <= residue_a < per_protomer and 0 <= residue_b < per_protomer):
                raise ValueError(
                    f"residue_a and residue_b must be 0-based indices within one "
                    f"protomer (0 to {per_protomer - 1}); "
                    f"got {residue_a} and {residue_b}.")

            pairs = [(residues[residue_a + p * per_protomer],
                      residues[residue_b + p * per_protomer])
                     for p in range(n_protomers)]

            start, stop = self._frame_bounds(universe)
            values, stamps = [], []
            for ts in universe.trajectory[start:stop:self.frame_step]:
                stamps.append(float(ts.time))
                per_pair = []
                for first, second in pairs:
                    distances = distance_array(
                        first.atoms.positions.astype(np.float32),
                        second.atoms.positions.astype(np.float32),
                        box=ts.dimensions)
                    switch = 1.0 / (1.0 + np.exp(
                        beta * (distances - lambda_constant * cutoff)))
                    per_pair.append(float(np.mean(switch)))
                values.append(float(np.mean(per_pair)))
            series.append(np.array(values))
            times.append(np.array(stamps))

        return {"times": times, "contacts": series}

---

## 6. Subunit decomposition and trend detection

In [ ]:
# ===========================================================================
#  Symmetry filter
#
#  Applied to the ASSEMBLY-WIDE contact matrix, before the subunit
#  decomposition below, so that every quantity derived from the contacts -
#  intra/inter averages, variances, trends, per-pair comparisons - comes from
#  one and the same filtered contact set.
# ===========================================================================

def apply_symmetry_filter(matrices, errors=None, n_protomers=None,
                          min_probability=None, min_copies=None):
    """Remove contacts that are not reproduced across the symmetric subunits.

    Parameters
    ----------
    matrices : array, shape (n_simulations, n, n)
        Assembly-wide contact probability matrices, n divisible by n_protomers.
    errors : array, same shape, optional
        Matching standard errors; filtered with the same mask so an error never
        survives a value that was removed.
    n_protomers : int
        Subunit count; defaults to N_PROTOMERS.
    min_probability : float
        A contact is "present" in a copy when its probability is at least this.
        Defaults to SYMMETRY_MIN_PROBABILITY.
    min_copies : int
        Number of the n_protomers copies in which the contact must be present to
        be kept. Defaults to SYMMETRY_MIN_COPIES, or to n_protomers if that is
        None (the strict Cn criterion).

    Returns
    -------
    filtered, filtered_errors, report
        Filtered copies of the inputs (the inputs are not modified) and a dict
        summarising what was removed, for the log and for the record.

    How it works
    ------------
    The matrix is viewed as subunit blocks, (subunit_i, residue, subunit_j,
    residue). For a fixed residue pair (i, j) and a fixed subunit RELATION -
    "same subunit" (shift 0), "next subunit" (shift 1), "opposite subunit"
    (shift 2), ... - there are exactly n_protomers symmetry-equivalent copies:
    block (k, k + shift) for every k. The filter compares those copies:

        present in >= min_copies copies  -> symmetric, kept as it is
        present in no copy               -> symmetric absence, kept as it is
        present in some, fewer than that -> ASYMMETRIC, set to zero in every copy

    Zeroing all copies, rather than only the odd ones out, is what makes the
    result a clean "remove this contact": averaging over subunits afterwards
    cannot resurrect it at a quarter of its value.

    Because a symmetric matrix's block (k, k + shift) is the transpose of block
    (k + shift, k), the masks for shift and n_protomers - shift are transposes
    of each other and the filtered matrix stays symmetric.
    """
    n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
    min_probability = (SYMMETRY_MIN_PROBABILITY if min_probability is None
                       else min_probability)
    if min_copies is None:
        min_copies = SYMMETRY_MIN_COPIES
    if min_copies is None:
        min_copies = n_protomers
    if not (1 <= min_copies <= n_protomers):
        raise ValueError(
            f"min_copies must be between 1 and n_protomers={n_protomers}, "
            f"got {min_copies}.")

    matrices = np.asarray(matrices, dtype=np.float32)
    if matrices.ndim != 3 or matrices.shape[1] != matrices.shape[2]:
        raise ValueError(
            f"expected (n_simulations, n, n), got {matrices.shape}")
    n = matrices.shape[1]
    if n % n_protomers != 0:
        raise ValueError(
            f"matrix size {n} is not divisible by n_protomers={n_protomers}. "
            "Either the selection did not cover whole subunits, or N_PROTOMERS "
            "is wrong.")
    per_protomer = n // n_protomers

    filtered = matrices.copy()
    filtered_errors = (None if errors is None
                       else np.asarray(errors, dtype=np.float32).copy())
    if filtered_errors is not None and filtered_errors.shape != filtered.shape:
        raise ValueError("errors must have the same shape as matrices.")

    for s in range(len(filtered)):
        # Views onto the s-th matrix as (subunit, residue, subunit, residue);
        # writing into them writes into `filtered` / `filtered_errors`.
        blocks = filtered[s].reshape(n_protomers, per_protomer,
                                     n_protomers, per_protomer)
        error_blocks = (None if filtered_errors is None else
                        filtered_errors[s].reshape(n_protomers, per_protomer,
                                                   n_protomers, per_protomer))
        for shift in range(n_protomers):
            # The n_protomers symmetry-equivalent copies of this relation.
            copies = np.stack([blocks[k, :, (k + shift) % n_protomers, :]
                               for k in range(n_protomers)], axis=0)
            n_present = (copies >= min_probability).sum(axis=0)
            asymmetric = (n_present > 0) & (n_present < min_copies)
            if not asymmetric.any():
                continue
            for k in range(n_protomers):
                blocks[k, :, (k + shift) % n_protomers, :][asymmetric] = 0.0
                if error_blocks is not None:
                    error_blocks[k, :, (k + shift) % n_protomers, :][asymmetric] = 0.0

    # Bookkeeping over the upper triangle of the assembly matrix (each unordered
    # pair once, diagonal excluded), pooled over all simulations.
    rows, cols = np.triu_indices(n, k=1)
    before = matrices[:, rows, cols]
    after = filtered[:, rows, cols]
    observed = int((before >= min_probability).sum())
    removed = int(((before != after) & (before >= min_probability)).sum())
    mass_before = float(before.sum())
    report = {
        "n_simulations": int(len(matrices)),
        "min_probability": float(min_probability),
        "min_copies": int(min_copies),
        "contacts_observed": observed,
        "contacts_removed": removed,
        "fraction_removed": (removed / observed) if observed else 0.0,
        "probability_mass_removed": ((mass_before - float(after.sum())) / mass_before
                                     if mass_before > 0 else 0.0),
    }
    return filtered, filtered_errors, report


# ===========================================================================
#  Subunit decomposition
#
#  A matrix over the whole assembly is reshaped into
#  (subunit_i, residue, subunit_j, residue) and averaged into three classes:
#
#     intra     subunit k with itself           - internal architecture
#     adjacent  subunit k with k+1 and k-1      - the main interface
#     diagonal  subunit k with k+2              - across the pore (tetramer+)
#
#  Averaging over equivalent subunit pairs is what makes a single per-protomer
#  matrix meaningful for a homo-oligomer.
# ===========================================================================

def decompose_by_subunit(matrices, n_protomers=None):
    """Split assembly-wide matrices into intra / adjacent / diagonal classes.

    `matrices` has shape (n_simulations, n, n) where n is divisible by
    n_protomers. Returns a dict of arrays each shaped
    (n_simulations, n_per_protomer, n_per_protomer).

    'diagonal' is only defined when there is a subunit two steps away, i.e.
    n_protomers >= 4; for a trimer it is returned as None.
    """
    n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
    matrices = np.asarray(matrices)
    if matrices.ndim != 3 or matrices.shape[1] != matrices.shape[2]:
        raise ValueError(
            f"expected (n_simulations, n, n), got {matrices.shape}")
    n = matrices.shape[1]
    if n % n_protomers != 0:
        raise ValueError(
            f"matrix size {n} is not divisible by n_protomers={n_protomers}. "
            "Either the selection did not cover whole subunits, or N_PROTOMERS "
            "is wrong.")
    per_protomer = n // n_protomers
    has_diagonal = n_protomers >= 4

    intra, adjacent, diagonal = [], [], []
    for matrix in matrices:
        blocks = matrix.reshape(n_protomers, per_protomer,
                                n_protomers, per_protomer)
        intra.append(np.mean([blocks[i, :, i, :]
                              for i in range(n_protomers)], axis=0))
        # Both orientations of the neighbour pair, so the result is symmetric
        # in a way a single orientation would not be.
        forward = np.mean([blocks[i, :, (i + 1) % n_protomers, :]
                           for i in range(n_protomers)], axis=0)
        backward = np.mean([blocks[(i + 1) % n_protomers, :, i, :]
                            for i in range(n_protomers)], axis=0)
        adjacent.append(0.5 * (forward + backward))
        if has_diagonal:
            diagonal.append(np.mean([blocks[i, :, (i + 2) % n_protomers, :]
                                     for i in range(n_protomers)], axis=0))

    return {
        "intra": np.asarray(intra, dtype=np.float32),
        "adjacent": np.asarray(adjacent, dtype=np.float32),
        "diagonal": (np.asarray(diagonal, dtype=np.float32)
                     if has_diagonal else None),
    }


def decompose_errors_by_subunit(errors, n_protomers=None):
    """As `decompose_by_subunit`, but propagating standard errors.

    Averaging k independent estimates divides the error by sqrt(k), so the
    combined error is sqrt(sum of squares) / k rather than the mean.
    """
    n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
    errors = np.asarray(errors)
    n = errors.shape[1]
    if n % n_protomers != 0:
        raise ValueError(
            f"matrix size {n} is not divisible by n_protomers={n_protomers}.")
    per_protomer = n // n_protomers
    has_diagonal = n_protomers >= 4

    def combine(blocks_list):
        stacked = np.stack(blocks_list, axis=0)
        return np.sqrt(np.sum(stacked ** 2, axis=0)) / len(blocks_list)

    intra, adjacent, diagonal = [], [], []
    for matrix in errors:
        blocks = matrix.reshape(n_protomers, per_protomer,
                                n_protomers, per_protomer)
        intra.append(combine([blocks[i, :, i, :] for i in range(n_protomers)]))
        neighbour = ([blocks[i, :, (i + 1) % n_protomers, :]
                      for i in range(n_protomers)]
                     + [blocks[(i + 1) % n_protomers, :, i, :]
                        for i in range(n_protomers)])
        adjacent.append(combine(neighbour))
        if has_diagonal:
            diagonal.append(combine([blocks[i, :, (i + 2) % n_protomers, :]
                                     for i in range(n_protomers)]))

    return {
        "intra": np.asarray(intra, dtype=np.float32),
        "adjacent": np.asarray(adjacent, dtype=np.float32),
        "diagonal": (np.asarray(diagonal, dtype=np.float32)
                     if has_diagonal else None),
    }


def inter_subunit_average(decomposed, n_protomers=None):
    """Average over all *other* subunits, weighted by how many there are.

    In a tetramer each subunit has two adjacent neighbours and one diagonal
    partner, giving (2 * adjacent + diagonal) / 3. The weights are derived from
    `n_protomers` rather than hard coded, so a trimer (no diagonal partner) and a
    pentamer (two diagonal partners) both come out right.
    """
    n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
    adjacent = decomposed["adjacent"]
    diagonal = decomposed["diagonal"]

    n_adjacent = 2 if n_protomers > 2 else 1
    n_other = n_protomers - 1
    n_diagonal = n_other - n_adjacent

    if n_diagonal <= 0 or diagonal is None:
        return adjacent
    return (n_adjacent * adjacent + n_diagonal * diagonal) / n_other


def pair_variance(matrices, threshold=None, min_separation=None):
    """Variance of each residue pair's value across a set of simulations.

    Returns (pairs, variance_matrix) where `pairs` is an (n, 2) integer array of
    upper-triangle indices whose variance exceeds `threshold` and whose sequence
    separation is at least `min_separation`.

    The variance is a population variance across the supplied simulations, so
    with only a handful of conditions it is a coarse measure - use it to rank
    pairs, not as a statistic to quote.
    """
    threshold = VARIANCE_THRESHOLD if threshold is None else threshold
    min_separation = (MIN_SEQUENCE_SEPARATION if min_separation is None
                      else min_separation)
    matrices = np.asarray(matrices, dtype=float)
    if matrices.shape[0] < 2:
        raise ValueError(
            f"Need at least two matrices to compute a variance, got "
            f"{matrices.shape[0]}.")

    variance = np.var(matrices, axis=0)
    upper = np.triu(np.ones_like(variance, dtype=bool), k=max(1, min_separation))
    rows, cols = np.where((variance >= threshold) & upper)
    return np.column_stack([rows, cols]).astype(int), variance


def monotonic_trend_pairs(matrices, x_values, tau_min=None, min_change=None,
                          p_floor=None, p_ceiling=None, min_separation=None,
                          symmetrise=True):
    """Find residue pairs whose value trends monotonically with `x_values`.

    Detection is rank-based (Kendall's tau) rather than least squares, because
    with only a handful of points one noisy value dominates an OLS slope. The
    magnitude is a Theil-Sen (median-of-pairwise-slopes) estimate, for the same
    reason.

    Parameters
    ----------
    matrices : array, shape (n_x, n_res, n_res)
        One matrix per value of `x_values`, in the same order.
    x_values : array, shape (n_x,)
        Strictly increasing, e.g. the temperatures.
    tau_min : float
        Minimum |Kendall tau|. This is the "how monotonic" knob; with 6 points,
        1.00 allows no inversions, about 0.87 allows one, about 0.73 allows two.
    min_change : float
        Minimum change across the full span, both observed (max - min) and
        fitted. Filters out monotonic but trivial drifts.
    p_floor, p_ceiling : float
        Drop pairs that never form (max < p_floor) or never break
        (min > p_ceiling); those are saturated, not trending.
    min_separation : int
        Minimum |i - j| along the sequence.

    Returns
    -------
    dict with keys 'increasing' and 'decreasing', each a dict of
    'pairs' (n, 2), 'tau', 'slope', 'change', 'first', 'last', sorted with the
    strongest trends first.
    """
    tau_min = TREND_TAU_MIN if tau_min is None else tau_min
    min_change = TREND_MIN_CHANGE if min_change is None else min_change
    p_floor = TREND_P_FLOOR if p_floor is None else p_floor
    p_ceiling = TREND_P_CEILING if p_ceiling is None else p_ceiling
    min_separation = (MIN_SEQUENCE_SEPARATION if min_separation is None
                      else min_separation)

    values = np.asarray(matrices, dtype=float)
    if values.ndim != 3 or values.shape[1] != values.shape[2]:
        raise ValueError(f"expected (n_x, n_res, n_res), got {values.shape}")
    n_x = values.shape[0]
    if n_x < 3:
        raise ValueError(
            f"need at least 3 points to speak of a trend, got {n_x}")

    x = np.asarray(x_values, dtype=float).ravel()
    if x.size != n_x:
        raise ValueError(
            f"len(x_values)={x.size} must match matrices.shape[0]={n_x}")
    if np.any(np.diff(x) <= 0):
        raise ValueError("x_values must be strictly increasing")

    if symmetrise:
        # Guards against a matrix that is only nominally symmetric.
        values = 0.5 * (values + np.transpose(values, (0, 2, 1)))

    rows, cols = np.triu_indices(values.shape[1], k=max(1, min_separation))
    series = values[:, rows, cols].T                      # (n_pairs, n_x)

    combos = np.array(list(combinations(range(n_x), 2)), dtype=int)
    a, b = combos[:, 0], combos[:, 1]
    differences = series[:, b] - series[:, a]
    x_differences = x[b] - x[a]

    # Kendall tau: (concordant - discordant) / n_combinations. Ties contribute
    # zero, so a flat series lands at tau = 0 and is filtered out below.
    tau = np.sign(differences).sum(axis=1) / combos.shape[0]
    slope = np.median(differences / x_differences, axis=1)
    change = slope * (x[-1] - x[0])

    minimum = np.nanmin(series, axis=1)
    maximum = np.nanmax(series, axis=1)
    keep = (np.isfinite(series).all(axis=1)
            & (maximum >= p_floor)
            & (minimum <= p_ceiling)
            & ((maximum - minimum) >= min_change)
            & (np.abs(change) >= min_change)
            & (np.abs(tau) >= tau_min))

    def pack(mask):
        index = np.flatnonzero(mask)
        order = np.lexsort((-np.abs(change[index]), -np.abs(tau[index])))
        index = index[order]
        return {
            "pairs": np.column_stack([rows[index], cols[index]]).astype(int),
            "tau": tau[index],
            "slope": slope[index],
            "change": change[index],
            "first": series[index, 0],
            "last": series[index, -1],
        }

    return {
        "increasing": pack(keep & (tau > 0) & (change > 0)),
        "decreasing": pack(keep & (tau < 0) & (change < 0)),
    }

---

## 7. Network plotting

In [ ]:
# ===========================================================================
#  Network-specific plotting
# ===========================================================================

def plot_network_matrix(matrix, residues, cbar_label, cmap="plasma",
                        log_scale=False, vmin=None, vmax=None, title=None,
                        figure_name=None):
    """Heat map of a residue-residue matrix, on real residue numbers.

    `log_scale` is usually right for contact probabilities, which span several
    orders of magnitude; leave it off for distances.
    """
    matrix = np.asarray(matrix, dtype=float)
    residues = np.asarray(residues)
    if matrix.shape[0] != len(residues):
        raise ValueError(
            f"matrix is {matrix.shape[0]}x{matrix.shape[1]} but the residue axis "
            f"has {len(residues)} entries.")

    fig, ax = plt.subplots(figsize=(13, 10))
    extent = [residues[0], residues[-1], residues[0], residues[-1]]

    if log_scale:
        positive = matrix[matrix > 0]
        if positive.size == 0:
            raise ValueError("All values are zero; cannot use a log colour scale.")
        low = float(positive.min()) if vmin is None else vmin
        high = float(matrix.max()) if vmax is None else vmax
        if high <= low:
            high = low * 10.0
        image = ax.imshow(np.clip(matrix, low, high), cmap=cmap, origin="lower",
                          aspect="equal", extent=extent,
                          norm=LogNorm(vmin=low, vmax=high))
        ticks, tick_labels = decade_ticks(low, high)
        cbar = fig.colorbar(image, ax=ax, ticks=ticks, pad=0.02)
        cbar.ax.set_yticklabels(tick_labels)
        cbar.minorticks_off()
    else:
        image = ax.imshow(matrix, cmap=cmap, origin="lower", aspect="equal",
                          extent=extent, vmin=vmin, vmax=vmax)
        cbar = fig.colorbar(image, ax=ax, pad=0.02)

    cbar.set_label(cbar_label)
    cbar.ax.tick_params(labelsize=FONTSIZE)
    ax.set_xlabel("Residue Number")
    ax.set_ylabel("Residue Number")
    apply_ticks(ax, "x", float(residues[0]), float(residues[-1]), n_ticks=7)
    apply_ticks(ax, "y", float(residues[0]), float(residues[-1]), n_ticks=7)
    if title:
        ax.set_title(title, fontsize=FONTSIZE)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_contact_arcs(matrix, pairs, residues, arc_height_scale=1.0,
                      max_arcs=200, vmin=None, vmax=None, log_scale=True,
                      min_linewidth=0.4, max_linewidth=1.8, title=None,
                      segments=None, show_domain_legend=False, figure_name=None):
    """Arc diagram of selected residue pairs along the sequence.

    `matrix` supplies the WEIGHT of every arc, and is normally the contact
    variance matrix returned by `pair_variance` alongside `pairs`. The diagram
    is then self-contained: which pairs are drawn, how dark each arc is and how
    thick it is all come from the same numbers, and no separate reference
    contact-probability matrix is needed. (Any non-negative matrix works, e.g.
    a probability matrix to show the strongest contacts instead.)

    Weights are mapped to darkness and thickness on a log scale - variances
    span several decades - between `vmin` and `vmax`, which default to the
    smallest and largest weight among the drawn pairs. Pass them explicitly to
    put several diagrams on one common scale. No colour key is drawn: the
    grey level is deliberately relative, a ranking within the figure, and the
    variance map alongside carries the absolute numbers.

    Arcs are stacked into non-overlapping levels so that nested contacts stay
    readable, and the weakest are drawn first so the strongest stay on top.
    The domain bands along the sequence axis are drawn in front of the arcs.

    `pairs` are 0-based indices into `matrix`; the x axis is labelled with the
    corresponding real residue numbers.
    """
    matrix = np.asarray(matrix, dtype=float)
    residues = np.asarray(residues)
    pairs = np.asarray(pairs, dtype=int).reshape(-1, 2)

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.set_xlabel("Residue Number")
    ax.set_xlim(float(residues[0]), float(residues[-1]))
    apply_ticks(ax, "x", float(residues[0]), float(residues[-1]), n_ticks=9)
    ax.set_yticks([])
    for side in ("left", "right", "top"):
        ax.spines[side].set_visible(False)

    # An empty pair list is a legitimate outcome - no pair passed the threshold -
    # so say so rather than raising on max() of an empty sequence.
    if pairs.size == 0:
        ax.text(0.5, 0.5, "No residue pairs passed the threshold",
                transform=ax.transAxes, ha="center", va="center",
                fontsize=FONTSIZE * 0.9, color="#666666")
        add_domain_bands(ax, segments=segments, show_legend=False)
        if figure_name:
            save_figure(fig, figure_name)
        return fig, ax

    # Unique unordered pairs with their weight; the diagonal has no arc.
    weights = {}
    for i, j in pairs:
        i, j = int(i), int(j)
        if i == j:
            continue
        if i > j:
            i, j = j, i
        weights[(i, j)] = max(float(matrix[i, j]), 0.0)
    if len(weights) > max_arcs:
        # Keep the heaviest arcs rather than an arbitrary slice.
        keep = sorted(weights, key=weights.get, reverse=True)[:max_arcs]
        weights = {pair: weights[pair] for pair in keep}
        print(f"  note: showing the {max_arcs} heaviest of "
              f"{len(pairs)} pair(s); raise max_arcs or VARIANCE_THRESHOLD")

    # Grey scale. On a log scale zero weights cannot be placed, so the lower end
    # defaults to the smallest POSITIVE weight and anything below is clipped.
    values = np.array(list(weights.values()))
    positive = values[values > 0]
    low = vmin
    if low is None:
        low = float(positive.min()) if (log_scale and positive.size) else float(values.min())
    high = float(values.max()) if vmax is None else vmax
    if high <= low:
        high = low * 10.0 if log_scale else low + 1.0e-12
    if log_scale:
        norm = LogNorm(vmin=low, vmax=high, clip=True)
    else:
        norm = Normalize(vmin=low, vmax=high, clip=True)
    # Light grey for the lightest arc, black for the heaviest.
    arc_cmap = LinearSegmentedColormap.from_list("arc_grey", ["0.85", "0.0"])

    # Stack arcs into levels: shortest span first, each into the lowest level
    # where it does not overlap an arc already there.
    ordered = sorted(weights, key=lambda pair: pair[1] - pair[0])
    levels, placement = [], {}
    for i, j in ordered:
        for level_index, level in enumerate(levels):
            if all(j <= a or i >= b for a, b in level):
                level.append((i, j))
                placement[(i, j)] = level_index
                break
        else:
            levels.append([(i, j)])
            placement[(i, j)] = len(levels) - 1

    residue_span = float(residues[-1] - residues[0])
    level_spacing = 0.06 * residue_span
    max_height = 0.0

    # Draw the lightest arcs first so the heaviest end up on top. All arcs sit
    # below zorder 1 so the domain bands, drawn afterwards at matplotlib's
    # default zorder, land in front of them.
    for (i, j) in sorted(weights, key=weights.get):
        strength = float(norm(weights[(i, j)]))          # 0 (lightest) .. 1 (darkest)
        left = float(residues[i])
        right = float(residues[j])
        width = right - left
        height = (level_spacing * (placement[(i, j)] + 1)
                  + width * arc_height_scale)
        max_height = max(max_height, height)
        ax.add_patch(Arc(
            ((left + right) / 2.0, 0.0), width=width, height=height,
            theta1=0.0, theta2=180.0,
            linewidth=min_linewidth + (max_linewidth - min_linewidth) * strength,
            color=arc_cmap(strength),
            zorder=0.5 + 0.4 * strength,
        ))

    # An Arc of `height` h rises only to h/2. Leaving the axis twice as tall as
    # the tallest arc keeps the diagram flat and wide rather than a dome that
    # fills the panel - the proportions the original diagrams had.
    ax.set_ylim(-0.02 * residue_span, max_height + level_spacing)
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    add_domain_bands(ax, segments=segments, show_legend=show_domain_legend)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_pair_comparison(matrices, errors, pairs, residues, x_values, xlabel,
                         colors=None, labels=None, title=None, n_columns=4,
                         figure_name=None):
    """One small panel per residue pair: value against `x_values`, with error bars.

    A grid of panels rather than a single crowded axis, because pairs sit at very
    different baseline probabilities and overlaying them hides the trends.
    """
    pairs = np.asarray(pairs, dtype=int).reshape(-1, 2)
    residues = np.asarray(residues)
    matrices = np.asarray(matrices, dtype=float)
    errors = None if errors is None else np.asarray(errors, dtype=float)

    if pairs.size == 0:
        print("  no pairs to compare")
        return None, None

    n_panels = len(pairs)
    n_columns = min(n_columns, n_panels)
    n_rows = int(np.ceil(n_panels / n_columns))
    fig, axes = plt.subplots(n_rows, n_columns,
                             figsize=(4.2 * n_columns, 3.6 * n_rows),
                             squeeze=False)
    colors = resolve_colors(colors, len(x_values))

    for panel, (i, j) in enumerate(pairs):
        ax = axes[panel // n_columns][panel % n_columns]
        values = matrices[:, i, j]
        bars = None if errors is None else errors[:, i, j]
        ax.errorbar(x_values, values, yerr=bars, color="#333333", marker="o",
                    markersize=6, linewidth=1.8, elinewidth=1.2, capsize=3)
        # Mark each condition in its own colour, matching the other notebooks.
        for x, value, color in zip(x_values, values, colors):
            ax.plot([x], [value], marker="o", markersize=9, color=color,
                    linestyle="None", zorder=4)
        ax.set_title(f"{int(residues[i])}–{int(residues[j])}",
                     fontsize=FONTSIZE * 0.7)
        ax.tick_params(labelsize=FONTSIZE * 0.55)
        ax.set_ylim(0.0, 1.0)
        ax.set_xticks(list(x_values))
        ax.set_xticklabels([f"{x:g}" for x in x_values],
                           fontsize=FONTSIZE * 0.55, rotation=45)
        ax.margins(x=0.08)

    for panel in range(n_panels, n_rows * n_columns):
        axes[panel // n_columns][panel % n_columns].axis("off")

    fig.supxlabel(xlabel, fontsize=FONTSIZE * 0.8)
    fig.supylabel("Contact Probability", fontsize=FONTSIZE * 0.8)
    if title:
        fig.suptitle(title, fontsize=FONTSIZE * 0.9)
    fig.tight_layout()

    if figure_name:
        save_figure(fig, figure_name)
    return fig, axes


def plot_contact_time_series(times, contacts, labels=None, colors=None,
                             legend_title=None, title=None, smooth=True,
                             window=51, polyorder=3, figure_name=None):
    """Contact of one residue pair through time, one line per simulation.

    Smoothing is for legibility only; the window is clamped to each trace, so a
    short series is handled without error.
    """
    fig, ax = plt.subplots(figsize=(15, 8))
    colors = resolve_colors(colors, len(contacts))
    labels = labels if labels is not None else [None] * len(contacts)

    for time, values, color, label in zip(times, contacts, colors, labels):
        time = np.asarray(time, dtype=float)
        values = np.asarray(values, dtype=float)
        if len(time) != len(values):
            raise ValueError(
                f"time has {len(time)} points but the series has {len(values)}.")
        if smooth and len(values) > polyorder + 2:
            local_window = min(window, len(values) - 1)
            if local_window % 2 == 0:
                local_window -= 1
            if local_window > polyorder:
                values = savgol_filter(values, local_window, polyorder)
        # Trajectory times are in ps; plot in ns.
        ax.plot(time * 1.0e-3, values, color=color, label=label, linewidth=2)

    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Contact")
    ax.set_ylim(0.0, 1.0)
    apply_ticks(ax, "y", 0.0, 1.0, n_ticks=6)
    x_lo = min(float(np.min(t)) for t in times) * 1.0e-3
    x_hi = max(float(np.max(t)) for t in times) * 1.0e-3
    if x_hi > x_lo:
        ax.set_xlim(x_lo, x_hi)
        apply_ticks(ax, "x", x_lo, x_hi, n_ticks=8)
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    if any(l is not None for l in labels):
        ax.legend(title=legend_title, loc="best", ncol=2,
                  fontsize=FONTSIZE * 0.7, title_fontsize=FONTSIZE * 0.7)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax

---

# Analysis

Everything above defines the pipeline; everything below runs it.

---

## 8. Build the networks

In [ ]:
# ===========================================================================
#  Build the networks
#
#  This is the expensive cell: one pass over every trajectory per model level.
# ===========================================================================

network = {}

for model_level in MODEL_LEVELS:
    protein_sel = PROTEIN_SELECTION[model_level]
    distance_sel = DISTANCE_SELECTION[model_level]
    cutoff = CONTACT_CUTOFF[model_level]
    heavy_only = HEAVY_ATOMS_ONLY[model_level]
    offset = RESIDUE_OFFSET.get(model_level, 0)

    print(f"\n{model_level} (contact cutoff {cutoff} A, "
          f"selection '{protein_sel}', distances on '{distance_sel}')")

    analysis = NetworkAnalysis(
        topologies=grid.paths(model_level, "topology"),
        trajectories=grid.paths(model_level, "trajectory"),
        labels=grid.labels(by="both"),
    )
    analysis.contact_network(protein_sel, cutoff=cutoff,
                             heavy_atoms_only=heavy_only)
    analysis.distance_network(distance_sel)

    # Symmetry filter FIRST, on the assembly-wide matrices, so that everything
    # derived below - the intra / inter averages, the variance maps, the trend
    # detection and the per-pair comparisons - comes from one filtered set.
    # The unfiltered matrices stay available as analysis.contacts.
    if APPLY_SYMMETRY_FILTER:
        assembly_contacts, assembly_errors, symmetry_report = apply_symmetry_filter(
            analysis.contacts, analysis.contact_errors)
        print(f"  symmetry filter: removed {symmetry_report['contacts_removed']} of "
              f"{symmetry_report['contacts_observed']} observed contacts "
              f"({100 * symmetry_report['fraction_removed']:.1f}%), "
              f"{100 * symmetry_report['probability_mass_removed']:.1f}% of the "
              f"total contact probability, pooled over "
              f"{symmetry_report['n_simulations']} conditions "
              f"(present >= {symmetry_report['min_probability']:g} in "
              f"< {symmetry_report['min_copies']} of {N_PROTOMERS} copies)")
    else:
        assembly_contacts = np.asarray(analysis.contacts)
        assembly_errors = np.asarray(analysis.contact_errors)
        symmetry_report = None
        print("  symmetry filter: OFF (APPLY_SYMMETRY_FILTER = False)")

    # Decompose into intra / adjacent / diagonal, for values and for errors.
    # Distances are geometry, not contacts, and are not filtered.
    contacts = decompose_by_subunit(assembly_contacts)
    contact_errors = decompose_errors_by_subunit(assembly_errors)
    distances = decompose_by_subunit(analysis.distances)

    # Average over all other subunits, weighted by how many of each kind there
    # are. The original code took the ERROR arrays from the VALUE dictionary
    # here, so every "error" was actually a copy of the contact probability -
    # error bars equal to the values themselves. Keeping the two dictionaries
    # distinct and named is what prevents that.
    inter_contacts = inter_subunit_average(contacts)
    inter_errors = inter_subunit_average(contact_errors)

    residues = analysis.residue_numbers(
        0, selection=protein_sel, heavy_atoms_only=heavy_only,
        residue_offset=offset)
    per_protomer = contacts["intra"].shape[1]
    residues_protomer = residues[:per_protomer]

    network[model_level] = {
        "analysis": analysis,
        "contacts": contacts,
        "contact_errors": contact_errors,
        "distances": distances,
        "inter_contacts": inter_contacts,
        "inter_errors": inter_errors,
        "residues": residues_protomer,
        "symmetry_report": symmetry_report,   # None when the filter is off
    }

    print(f"  {len(analysis.contacts)} conditions | "
          f"{len(residues)} residues total | {per_protomer} per protomer")
    print(f"  mean intra contact probability : "
          f"{contacts['intra'].mean():.4f}")
    print(f"  mean inter-subunit probability : {inter_contacts.mean():.4f}")
    print(f"  median intra error             : "
          f"{np.median(contact_errors['intra']):.5f}")

print("\nDone.")

---

## 9. Distance and contact maps

In [ ]:
# Distance and contact maps for one representative condition per model level.
#
# Set SHOW_ALL_CONDITIONS = True to loop over the whole grid - that is 18 figures
# per model level, which is a lot to scroll through.
SHOW_ALL_CONDITIONS = False
REPRESENTATIVE = {"temperature_k": 310, "salinity_mm": 150}

for model_level, entry in network.items():
    indices = (range(len(grid)) if SHOW_ALL_CONDITIONS
               else grid.where(**REPRESENTATIVE))
    residues = entry["residues"]
    for index in indices:
        plot_network_matrix(
            entry["distances"]["intra"][index], residues,
            cbar_label=r"Distance ($\mathrm{\AA}$)", cmap="YlOrRd",
            title=f"{model_level.title()} {grid[index]}: intra-subunit distances",
            figure_name=f"distance_intra_{model_level}_{index}",
        )
        plt.show()

        plot_network_matrix(
            entry["contacts"]["intra"][index], residues,
            cbar_label="Contact Probability", cmap="plasma", log_scale=True,
            title=f"{model_level.title()} {grid[index]}: intra-subunit contacts",
            figure_name=f"contact_intra_{model_level}_{index}",
        )
        plt.show()

        plot_network_matrix(
            entry["inter_contacts"][index], residues,
            cbar_label="Contact Probability", cmap="plasma", log_scale=True,
            title=f"{model_level.title()} {grid[index]}: inter-subunit contacts",
            figure_name=f"contact_inter_{model_level}_{index}",
        )
        plt.show()

---

## 10. Contact variance across conditions

In [ ]:
# Which residue pairs change most across temperature, at each salinity?
#
# The variance map shows where the change is; the arc diagram shows the same
# pairs drawn along the sequence, which is easier to relate to the domains.
# Both are built from the symmetry-filtered contacts (see the build cell), and
# the arc diagram takes its darkness and thickness from the variance matrix
# itself, so the two figures are two views of exactly the same numbers.
variance = {}

for model_level, entry in network.items():
    residues = entry["residues"]
    variance[model_level] = {}

    for salinity in SALINITIES_MM:
        indices = grid.where(salinity_mm=salinity)

        for scope, matrices in (("intra", entry["contacts"]["intra"]),
                                ("inter", entry["inter_contacts"])):
            pairs, variance_matrix = pair_variance(matrices[indices])
            variance[model_level][(scope, salinity)] = (pairs, variance_matrix)
            print(f"{model_level:<10} {scope:<6} {salinity:>4.0f} mM: "
                  f"{len(pairs):5d} pair(s) above variance "
                  f"{VARIANCE_THRESHOLD:g}")

    # Plot one salinity to keep the output manageable.
    salinity = SALINITIES_MM[-1]
    for scope in ("intra", "inter"):
        pairs, variance_matrix = variance[model_level][(scope, salinity)]
        plot_network_matrix(
            variance_matrix, residues,
            cbar_label="Contact Probability Variance", cmap="plasma",
            log_scale=True,
            title=f"{model_level.title()} {scope}-subunit, {salinity:.0f} mM: "
                  "variance across temperature",
            figure_name=f"variance_{scope}_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

        # The variance matrix weights the arcs: a darker, thicker arc is a
        # pair whose contact changes more across temperature.
        plot_contact_arcs(
            matrix=variance_matrix,
            pairs=pairs, residues=residues, arc_height_scale=0.6,
            title=f"{model_level.title()} {scope}-subunit, {salinity:.0f} mM: "
                  "most variable contacts",
            figure_name=f"arcs_{scope}_{model_level}_{salinity:.0f}mM",
        )
        plt.show()

---

## 11. Monotonic temperature trends

In [ ]:
# Residue pairs whose contact probability trends monotonically with temperature.
#
# These are the candidates for contacts that make or break as the channel
# responds to heat. Detection is rank-based, so a single noisy temperature does
# not manufacture a trend.
#
# Both scopes are scanned. They answer different questions: an intra-subunit
# trend is a conformational change inside one protomer, while an inter-subunit
# one is the interface itself loosening or tightening.
trends = {}

for model_level, entry in network.items():
    residues = entry["residues"]
    trends[model_level] = {}
    print(f"\n{'=' * 78}\n{model_level.upper()}"
          f"  (|tau| >= {TREND_TAU_MIN}, change >= {TREND_MIN_CHANGE})"
          f"\n{'=' * 78}")

    for scope, matrices in (("intra", entry["contacts"]["intra"]),
                            ("inter", entry["inter_contacts"])):
        for salinity in SALINITIES_MM:
            indices = grid.where(salinity_mm=salinity)
            result = monotonic_trend_pairs(matrices[indices],
                                           x_values=TEMPERATURES_K)
            trends[model_level][(scope, salinity)] = result

            n_up = len(result["increasing"]["pairs"])
            n_down = len(result["decreasing"]["pairs"])
            print(f"\n{scope}-subunit, {salinity:.0f} mM: "
                  f"{n_up} forming, {n_down} breaking")

            for direction in ("increasing", "decreasing"):
                block = result[direction]
                if len(block["pairs"]) == 0:
                    continue
                verb = "forming" if direction == "increasing" else "breaking"
                print(f"  strongest {verb}:")
                print(f"    {'pair':>13}{'tau':>8}{'change':>9}"
                      f"{'P(low T)':>10}{'P(high T)':>11}   domains")
                for rank in range(min(5, len(block["pairs"]))):
                    i, j = block["pairs"][rank]
                    res_i, res_j = int(residues[i]), int(residues[j])
                    print(f"    {res_i:5d}-{res_j:<7d}"
                          f"{block['tau'][rank]:+8.2f}"
                          f"{block['change'][rank]:+9.3f}"
                          f"{block['first'][rank]:10.3f}"
                          f"{block['last'][rank]:11.3f}"
                          f"   {describe_domain(res_i)} / {describe_domain(res_j)}")

---

## 12. Per-pair comparison

In [ ]:
# Detailed comparison of specific residue pairs across temperature.
#
# When RESIDUE_PAIRS_OF_INTEREST is None the notebook uses the strongest
# trending pairs it just found. That is deliberate: a hand-maintained list of
# pairs goes stale as soon as the data changes, and the original notebook had
# exactly that problem - its pair list and its axis labels described different
# residues.
COMPARISON_SALINITY = SALINITIES_MM[-1]
COMPARISON_SCOPE = "intra"        # "intra" or "inter"

for model_level, entry in network.items():
    residues = entry["residues"]
    indices = grid.where(salinity_mm=COMPARISON_SALINITY)
    result = trends[model_level][(COMPARISON_SCOPE, COMPARISON_SALINITY)]
    matrices = (entry["contacts"]["intra"] if COMPARISON_SCOPE == "intra"
                else entry["inter_contacts"])
    error_matrices = (entry["contact_errors"]["intra"]
                      if COMPARISON_SCOPE == "intra" else entry["inter_errors"])

    if RESIDUE_PAIRS_OF_INTEREST is not None:
        # Translate real residue numbers into indices into the matrices.
        lookup = {int(r): k for k, r in enumerate(residues)}
        pairs = []
        for res_i, res_j in RESIDUE_PAIRS_OF_INTEREST:
            if res_i not in lookup or res_j not in lookup:
                print(f"  skipping {res_i}-{res_j}: outside the residue range "
                      f"{int(residues[0])}-{int(residues[-1])}")
                continue
            pairs.append([lookup[res_i], lookup[res_j]])
        pairs = np.asarray(pairs, dtype=int).reshape(-1, 2)
        source = "RESIDUE_PAIRS_OF_INTEREST"
    else:
        both = [result["decreasing"]["pairs"], result["increasing"]["pairs"]]
        both = [p for p in both if len(p)]
        pairs = (np.vstack(both)[:N_AUTO_PAIRS] if both
                 else np.empty((0, 2), dtype=int))
        source = f"the {len(pairs)} strongest trending pair(s)"

    print(f"{model_level}: comparing {source} "
          f"({COMPARISON_SCOPE}-subunit, {COMPARISON_SALINITY:.0f} mM)")

    plot_pair_comparison(
        matrices=matrices[indices],
        errors=error_matrices[indices],
        pairs=pairs, residues=residues, x_values=list(TEMPERATURES_K),
        xlabel="Temperature (K)",
        title=f"{model_level.title()}: {COMPARISON_SCOPE}-subunit contacts, "
              f"{COMPARISON_SALINITY:.0f} mM",
        figure_name=f"pair_comparison_{COMPARISON_SCOPE}_{model_level}_"
                    f"{COMPARISON_SALINITY:.0f}mM",
    )
    plt.show()

---

## 13. Contact time series

In [ ]:
# Contact of a single residue pair through time, across temperatures.
#
# The matrices above give a time average; this shows whether a contact is stable,
# intermittent, or breaks at a particular moment - three situations that can all
# produce the same mean.
TIME_SERIES_SALINITY = SALINITIES_MM[-1]
TIME_SERIES_SCOPE = "intra"       # "intra" or "inter"

for model_level, entry in network.items():
    residues = entry["residues"]
    result = trends[model_level][(TIME_SERIES_SCOPE, TIME_SERIES_SALINITY)]

    # Follow the single strongest breaking contact, falling back to a forming one.
    for direction in ("decreasing", "increasing"):
        if len(result[direction]["pairs"]):
            residue_i, residue_j = result[direction]["pairs"][0]
            break
    else:
        print(f"{model_level}: no trending pair to follow.")
        continue

    indices = grid.where(salinity_mm=TIME_SERIES_SALINITY)
    subset = NetworkAnalysis(
        topologies=grid.paths(model_level, "topology", indices),
        trajectories=grid.paths(model_level, "trajectory", indices),
        labels=grid.labels(indices, by="temperature"),
    )
    series = subset.residue_pair_time_series(
        int(residue_i), int(residue_j),
        selection=PROTEIN_SELECTION[model_level],
        cutoff=CONTACT_CUTOFF[model_level],
    )

    plot_contact_time_series(
        series["times"], series["contacts"],
        labels=grid.labels(indices, by="temperature"),
        legend_title="Temperature",
        title=f"{model_level.title()}: residues "
              f"{int(residues[residue_i])}–{int(residues[residue_j])} "
              f"at {TIME_SERIES_SALINITY:.0f} mM",
        figure_name=f"contact_time_series_{model_level}",
    )
    plt.show()

---

## Troubleshooting

**`the selection '...' matched no atoms`**
`"protein"` relies on standard amino-acid resnames, which coarse-grained
topologies often lack. Inspect what is there and set `PROTEIN_SELECTION` /
`DISTANCE_SELECTION` accordingly:

```python
u = mda.Universe(str(grid.paths("coarse", "topology")[0]))
print(sorted(set(u.residues.resnames)))
print(sorted(set(u.atoms.names)))
```

**`matrix size N is not divisible by n_protomers`**
The selection did not cover whole subunits, or `N_PROTOMERS` is wrong. Note the
contact matrix is over *residues* of `PROTEIN_SELECTION` while the distance
matrix is over *atoms* of `DISTANCE_SELECTION`; both must divide evenly, so
`DISTANCE_SELECTION` must pick exactly one atom per residue.

**`warning: ... uses only N block(s)`**
`N_BLOCKS` was reduced because there are too few frames. Lower `FRAME_STEP` so
more frames are read, or accept that the error bars are unreliable.

**Every error bar is suspiciously small, or exactly zero**
Zero means only one non-empty block survived. Very small usually means too few
blocks, or blocks shorter than the correlation time of the contact. Increase the
trajectory length or `N_BLOCKS`.

**The arc diagram is a solid mass of arcs**
Raise `VARIANCE_THRESHOLD`, or `MIN_SEQUENCE_SEPARATION` to drop near-neighbour
contacts. `max_arcs` caps the drawing at the heaviest arcs and prints a note
when it does.

**All the arcs look the same shade of grey**
The grey scale is relative to the drawn pairs (lightest = smallest variance
among them, black = largest). If the variances all lie within a factor of two
of each other there is little to distinguish; pass `vmin`/`vmax` to
`plot_contact_arcs` to fix the scale, e.g. `vmin=VARIANCE_THRESHOLD, vmax=0.1`,
which also makes different diagrams comparable.

**The symmetry filter removed most of the contacts**
Check the "symmetry filter" line printed by the build cell. A very high removal
fraction usually means `SYMMETRY_MIN_PROBABILITY` is too low (single-frame
contacts counted as "present" in one subunit) or the sampling is too short for
the subunits to have converged. Relax `SYMMETRY_MIN_COPIES` (e.g. 3 of 4) before
lowering the probability threshold, and compare with `APPLY_SYMMETRY_FILTER =
False`.

**No pairs pass the trend filters**
In order: lower `TREND_TAU_MIN` (0.73 tolerates two inversions out of six
points), then lower `TREND_MIN_CHANGE`. If still nothing, there may be no
monotonic response — a real and reportable result.

**It is unbearably slow**
Cost is dominated by the contact network: frames x atoms. Raise `FRAME_STEP`,
keep `HEAVY_ATOMS_ONLY` True for atomistic runs, and consider a protein-only
trajectory if you are not already using one.

---

## Notes on method

- **A contact is per residue pair, per frame, any-atom.** Many atom-level
  contacts between the same two residues count once. This is occupancy, not
  interaction strength.
- **The diagonal is set to 1.0 by convention.** It is cosmetic, and it means the
  diagonal must be excluded from any statistic over the matrix — which is why
  `MIN_SEQUENCE_SEPARATION` defaults to more than zero.
- **Errors are block standard errors**, and they assume blocks are independent.
  If a contact's correlation time is longer than a block, the error is still
  underestimated. Treat them as a lower bound.
- **The symmetry filter runs first, on the assembly-wide matrix.** A contact
  present in some but not all of its `N_PROTOMERS` symmetry-equivalent copies is
  set to zero in every copy; symmetric presence and symmetric absence are left
  alone. Errors are masked identically. Every figure downstream uses the same
  filtered set, so a pair that appears in the variance map is the same pair the
  trend detection and the per-pair comparison see. The time-series cell reads
  the trajectory directly and is not filtered, but the pair it follows is chosen
  from the filtered results.
- **Arc darkness and thickness are the variance**, mapped on a log scale between
  the smallest and largest variance among the drawn pairs. The scale is relative
  by design (no colour key); read absolute values off the variance map.
- **The subunit decomposition assumes a homo-oligomer** whose subunits appear as
  consecutive equal-length residue blocks. For a hetero-oligomer set
  `N_PROTOMERS = 1` and slice the matrix manually.
- **`inter_subunit_average` weights adjacent and diagonal contributions** by how
  many partners of each kind a subunit has, derived from `N_PROTOMERS`. For a
  trimer there is no diagonal partner and the adjacent average is returned
  unchanged.
- **Kendall's tau, not a linear fit.** With six temperatures an OLS slope is
  dominated by whichever point is noisiest. Tau asks only about ordering, and the
  Theil-Sen slope gives a magnitude that survives one bad point.
- **Variance across conditions is a ranking device**, not a statistic to quote:
  with six conditions it has very few degrees of freedom.
- **The smooth switching function** in the time series avoids the flicker a hard
  cutoff produces for a pair sitting near the cutoff distance. It is a
  presentation choice; `beta` and `lambda_constant` change the numbers, so keep
  them fixed when comparing runs.

---

## Repository housekeeping

This notebook is one of several in the repository, so a few conventions keep
them from interfering with each other.

**Shared, repository-level files.** `requirements.txt` and `.gitignore` live at
the repository root, one of each, covering every notebook. Neither can be
renamed per notebook: git only recognises the filename `.gitignore`, and
`pip install -r requirements.txt` is what every reader and CI runner expects.

**Per-notebook outputs.** Each notebook writes to `results/<NOTEBOOK_SLUG>/`,
set in the configuration cell, so two notebooks saving a figure of the same name
cannot overwrite one another.

**Shared input data.** `DATA_ROOT` and `demo_data/` are deliberately *not*
namespaced, so sibling notebooks read the same trajectories rather than each
keeping a copy.

**Sections 1-4 of the configuration cell are identical in every notebook.** If
you change the simulation grid, the protomer count or the domain boundaries,
change them the same way in each notebook. If that becomes tedious, move those
sections into a `channel_config.py` at the repository root and import it from
each notebook instead.

**Outputs are stripped** from the committed notebooks, which keeps diffs small
and stops absolute data paths leaking through cell output:

```bash
pip install nbstripout
nbstripout --install     # run once, inside the repository
```
